The Core Conceptual Problem
Mamba-1 parameterizes selective State Space Models (SSMs) via a 1st-order continuous-time differential equation discretized using Exponential-Euler:

While Mamba-1 achieved $O(T)$ linear-time inference memory, its sequential recurrence was sequential on GPU SRAM ($O(T)$ scan sequential dependency).

+-----------------------------------------------------------------------------------+
|                            MAMBA-2 BOTTLENECK DUALITY                             |
+-----------------------------------------------------------------------------------+
| 1. HARDWARE CEILING (Decoding Regime):                                            |
|    - Memory-Bound Arithmetic Intensity during autoregressive generation ($T=1$).  |
|    - Scalar update h_t = alpha_t * h_{t-1} + B_t * x_t yields high bandwidth     |
|      overhead relative to compute FLOPs (Low HBU/MFU on Tensor Cores).            |
|                                                                                   |
| 2. EXPRESSIVITY CEILING (State Tracking & Recall):                                |
|    - Real-valued diagonal decay causes monotone state attenuation.                |
|    - Zero rotational phase degrees of freedom -> Inability to solve Parity /      |
|      Modular Arithmetic / Associative Memory (Induction Head emulation).          |
+-----------------------------------------------------------------------------------+

Mathematical Derivations & State DynamicsA. The Mamba-2 (SSD) Recurrence FormulationIn Mamba-2, the state update for head $h$ at token step $t$ is expressed as:$$h_t = \alpha_t h_{t-1} + B_t x_t \quad \in \mathbb{R}^N$$$$y_t = C_t^\top h_t \quad \in \mathbb{R}$$where:$\alpha_t = \exp(\Delta_t A_t) \in (0, 1]$ is a real scalar decay factor.$B_t, C_t \in \mathbb{R}^N$ are data-dependent projection vectors derived from input $x_t$.$h_t \in \mathbb{R}^N$ is the latent memory state vector.